# 00 — Environment & Reproducibility

**CuSelect-AI · Chapter I — Foundations**

**Research question:** *Can we run a controlled Python scientific environment and reproduce our starting Cu(100)/Cu–Ag geometries without relying on hidden notebook state?*

This notebook is our laboratory notebook for the **computational environment**, not a scientific result about ethylene selectivity. It must run cleanly before any literature, ML or AI-agent experiments.

**Inputs:** packaged Python modules, `pyproject.toml`, `configs/CU-001_protocol.json`.

**Outputs:** `outputs/reports/00_environment_snapshot.json`, visible geometry smoke-test results.

**Acceptance gate:** correct Python/kernel, imports, geometry consistency assertions, saved snapshot. Continue to Notebook 01 only when all pass.


## 0.1 — Why reproducibility comes first

Imagine a scientific colleague receives our notebook. They should reproduce the same *model geometry* without knowing which cells we ran earlier. The notebook should use reusable functions from `src/cuselect/`, not copy and paste the code that constructs surfaces.

**Rules:** run cells top to bottom; do not silently suppress warnings; distinguish modeled, measured, reported and unknown; never store credentials or private computer identifiers in an output.


In [1]:
# PSEUDOCODE
# 1. Locate the current project folder.
# 2. Confirm we opened JupyterLab from the repository root.
# 3. Report the active Python interpreter and architecture.

from pathlib import Path
import platform
import sys

from cuselect.project_checks import require_project_root, environment_snapshot

# Move from the notebooks folder to the project root

ROOT = require_project_root(Path.cwd().parent)
print('Project root:', ROOT.name)
print('Python:', sys.version.split()[0])
print('Kernel interpreter:', sys.executable)
print('Operating system:', platform.system(), '| CPU architecture:', platform.machine())
assert sys.version_info[:2] == (3, 11), 'Please select the Python (CuSelect-AI) kernel with Python 3.11.'


Project root: CuSelect-AI
Python: 3.11.18
Kernel interpreter: /opt/anaconda3/envs/cuselect/bin/python
Operating system: Darwin | CPU architecture: arm64


## 0.2 — Check our minimal scientific toolkit

Before advanced AI models, we need trusted numerical arrays, data tables, atomistic structures, and typed data validation. Version reporting prevents us from comparing outputs from unrecorded software environments.


In [2]:
# PSEUDOCODE
# 1. Import the small trusted scientific stack.
# 2. Record installed package versions without reading any secrets.
# 3. Assert that required project dependencies are available.

import ase
import numpy as np
import pandas as pd
import pydantic

snapshot = environment_snapshot()
required = ['ase', 'numpy', 'pandas', 'pydantic']
assert all(snapshot['packages'][name] != 'not_installed' for name in required)

for package, version in snapshot['packages'].items():
    print(f'{package:12} {version}')


ase          3.29.0
numpy        2.4.6
pandas       2.3.3
pydantic     2.14.2
jupyterlab   4.6.5
ipykernel    7.4.0
streamlit    1.66.1
pytest       9.2.0


## 0.3 — Read the frozen scientific question

The versioned protocol is stored outside the notebook. If our research question, assumptions or primary metric changes later, we will edit the protocol under version control and document the decision. We do not silently redefine it in multiple notebooks.


In [3]:
# PSEUDOCODE
# 1. Open the campaign protocol.
# 2. Validate its data types and required fields.
# 3. Display the campaign ID and target without changing anything.

from cuselect.protocol import load_research_protocol

protocol = load_research_protocol(ROOT / 'configs' / 'CU-001_protocol.json')
print('Campaign:', protocol.campaign_id)
print('Research target:', protocol.system.target_product)
print('Scientific objective:', protocol.research_question)
assert protocol.known_model_boundaries.selectivity_predicted is False


Campaign: CU-001
Research target: ethylene (C2H4)
Scientific objective: Can a mechanism-aware scientific critic reduce unsupported advancement of Cu-Ag CO2RR catalyst proposals without excessive rejection of valid candidates?


## 0.4 — First atomistic smoke test: structure, not catalysis

We deliberately reuse the Stage-1 reference structures: a 3 × 3 × 4 Cu(100) slab and a slab with a single substitutional top-layer Ag atom. These **unrelaxed, periodically repeated geometries** are useful for testing our software; they do **not** establish surface stability, selectivity, reaction barriers, or Faradaic efficiency.

**Predicted output:** 36 atoms per slab; `Cu36` and `AgCu35`; periodic in x and y only. Checks must fail loudly if they are inconsistent.


In [4]:
# PSEUDOCODE
# 1. Construct two geometries using tested library functions.
# 2. Confirm atom counts, formulas and periodic boundary conditions.
# 3. Inspect summarized geometry-only metadata.

from cuselect.structures import make_cu100, make_substitutional_cuag, geometry_summary

reference = make_cu100()
substitution = make_substitutional_cuag()

assert reference.get_chemical_formula() == 'Cu36'
assert substitution.get_chemical_formula() == 'AgCu35'
assert len(reference) == len(substitution) == 36
assert list(reference.pbc) == [True, True, False]
assert list(substitution.pbc) == [True, True, False]
assert substitution.info['geometry_status'] == 'unrelaxed'

pd.DataFrame([geometry_summary(reference), geometry_summary(substitution)])


,candidate_id,chemical_formula,total_atoms,top_layer_atoms,ag_atoms,ag_fraction_top_layer,periodic_xyz,geometry_status,energy_status,selectivity_status
0,CU-001-REF,Cu36,36,9,0,0.0000,"[True, True, False]",unrelaxed,not_computed,not_predicted
1,CU-001-AG1,AgCu35,36,9,1,0.1111,"[True, True, False]",unrelaxed,not_computed,not_predicted


## 0.5 — Export a minimal environment record

A research record must capture the software context, but not collect usernames, serial numbers, API keys, environment variables or other unnecessary personal information. The environment report is generated and excluded from Git; the notebook itself is version-controlled.


In [5]:
# PSEUDOCODE
# 1. Create an outputs/reports directory if necessary.
# 2. Write the package/version snapshot as human-readable JSON.
# 3. Read the file back to verify it was saved correctly.

import json

report_dir = ROOT / 'outputs' / 'reports'
report_dir.mkdir(parents=True, exist_ok=True)
report_path = report_dir / '00_environment_snapshot.json'
report_path.write_text(json.dumps(snapshot, indent=2) + '\n', encoding='utf-8')

reloaded = json.loads(report_path.read_text(encoding='utf-8'))
assert reloaded['python'] == snapshot['python']
print('Saved:', report_path.relative_to(ROOT))


Saved: outputs/reports/00_environment_snapshot.json


## 0.6 — Interpretation and limitations

**What we have established:** the project package imports, the protocol can be loaded, and two fixed geometry models have the intended composition, atom counts and boundary conditions. An environment report has been written.

**What we have NOT established:** catalyst stability, adsorption energies, C–C coupling pathways, product selectivity, model error or scientific novelty. Those require later notebooks and independent evidence.

### Handoff → Notebook 01

- **Input:** `configs/CU-001_protocol.json` and bibliographic seed register `data/papers.csv`.
- **What to do next:** make our research objective, assumptions and validation rules explicit.
- **Transition gate:** restart kernel, choose **Run All**, and ensure this notebook completes without assertion errors. Run `python -m pytest -q` in Terminal as an independent check.
